# Water Systems Planner — Equations &amp; Formulas

A reference for every engineering formula behind **`water-systems-planner.html`**
(the *Uplift Off-Grid Systems Planner*). The app is a single-file browser tool;
this notebook lifts its calculation core out of the JavaScript and restates each
model in the notebook house style — a **markdown cell** stating the model,
assumptions and equations (LaTeX), immediately followed by a **standalone code
cell** that declares `UPPER_SNAKE_CASE` inputs, computes the derived quantities,
and ends with a formatted `print(...)` report.

Each code cell reproduces the app's numbers using its **shipped default inputs**
(the `S` state object, `fixtures`, `loads`, and the constants blocks in the HTML),
so the printed figures should match the running app. Cells are self-contained —
each imports `math` and defines its own helpers; none depend on prior cell state.

**Units are mixed and explicit by design.** The app does *all internal math in
imperial* (°F, gal, GPM, ft, in, ft², Btu/hr) and converts only at the display
boundary; urine volume is the lone exception (stored in litres). This notebook
follows the same convention.

### What's modeled (by tab)

| Domain | Core formulas |
|---|---|
| **Constants &amp; units** | water density, Btu↔Wh, psi↔ft, gal↔in³ |
| **Water demand &amp; tanks** | fixture demand, hot fraction, peak flow, autonomy sizing |
| **Hot water** | reheat energy, recovery time, standby loss, shower capacity, tank geometry |
| **Drainage (DWV)** | fixture-unit (DFU) totals → branch/main/vent sizing |
| **Composting toilet** | urine/leachate volume → container sizing |
| **Pump sizing** | Hazen–Williams friction, Total Dynamic Head, velocity, hydraulic &amp; electrical power, curve operating point |
| **Power balance** | water-system daily energy vs budget |
| **Battery** | usable energy, autonomy days, modules-for-target |
| **Solar** | array output, load coverage, half-sine daily profile |
| **Thermal envelope** | steel-bridged assembly R → UA → U₀ → HUD zone |
| **Thermal loads** | heating (conduction + infiltration), cooling (sol-air + solar + latent) |
| **HVAC &amp; seasonal** | COP curves, electrical draw, degree-day annual energy |

> Source of record: `c:\repos\systems\water-systems-planner.html`. Physics
> constants and reference values (COP curves, sol-air uplifts, HUD caps) live in
> that file's *Assumptions backend*; the values below match its shipped defaults.


## 1 · Global constants &amp; unit conversions

Every downstream formula rests on a handful of physical constants and conversion
factors. These are the literals scattered through the app (`heatWh`, `MASS`,
`hwKwh`, the `u*` converters); collected here once.

| Symbol | Value | Meaning |
|---|---|---|
| $\rho_w$ | 8.34 lb/gal (1 kg/L) | density of water |
| $c_p$ | 1 Btu/(lb·°F) | specific heat of water (implicit) |
| — | 3.412 Btu/Wh | energy: Btu/hr ↔ W (and Btu ↔ Wh) |
| $\text{BTUH\_W}$ | 0.293071 W/(Btu/hr) | inverse of the above (Btu/hr → W) |
| — | 3412 Btu/kWh | energy: Btu ↔ kWh |
| — | 231 in³/gal | volume |
| $\text{L2GAL}$ | 0.264172 gal/L | volume |
| $\text{PSI2FT}$ | 2.3067 ft/psi | pressure head of water |
| — | 3.78541 L/gal | volume (metric display) |

**Water heating energy** — heating a mass of water $m$ (lb) by $\Delta T$ (°F)
takes $Q = m\,c_p\,\Delta T$ Btu; with $c_p=1$ and $m=8.34\,V_{\text{gal}}$:

$$Q_{\text{Wh}} = \frac{8.34 \cdot V_{\text{gal}} \cdot \Delta T}{3.412}
\qquad\text{(app's \texttt{heatWh}), or}\qquad
Q_{\text{kWh}} = \frac{8.34 \cdot V_{\text{gal}} \cdot \Delta T}{3412}$$


In [1]:
# ── Global constants & unit conversions ─────────────────────────────────────
import math

# Physical constants (imperial internal units)
RHO_W_LB_GAL = 8.34       # lb/gal   density of water (≈ 1 kg/L)
BTU_PER_WH   = 3.412      # Btu/Wh   (also Btu/hr per W)
BTUH_W       = 0.293071   # W per Btu/hr   (= 1/3.412, the app's BTUH_W)
BTU_PER_KWH  = 3412       # Btu/kWh
IN3_PER_GAL  = 231        # in³/gal
L2GAL        = 0.264172   # gal/L
PSI2FT       = 2.3067     # ft of water head per psi
GAL_TO_L     = 3.78541    # L/gal

def heat_wh(gal, dT_F):
    # Energy [Wh] to raise `gal` of water by dT_F degrees F (app's heatWh)
    return RHO_W_LB_GAL * gal * dT_F / BTU_PER_WH

def heat_kwh(gal, dT_F):
    # Energy [kWh] for the same (app's hwKwh)
    return RHO_W_LB_GAL * gal * dT_F / BTU_PER_KWH

print("Global constants")
print(f"  Water density          : {RHO_W_LB_GAL} lb/gal (1 kg/L)")
print(f"  Btu <-> Wh             : {BTU_PER_WH} Btu/Wh   |  Btu/hr -> W : {BTUH_W}")
print(f"  Pressure head          : {PSI2FT} ft/psi")
print(f"  Volume                 : {IN3_PER_GAL} in3/gal, {L2GAL} gal/L")
print()
print("Worked example — heat 10 gal from 64F to 140F (dT = 76 F):")
print(f"  {heat_wh(10, 76):.0f} Wh  =  {heat_kwh(10, 76):.3f} kWh")

Global constants
  Water density          : 8.34 lb/gal (1 kg/L)
  Btu <-> Wh             : 3.412 Btu/Wh   |  Btu/hr -> W : 0.293071
  Pressure head          : 2.3067 ft/psi
  Volume                 : 231 in3/gal, 0.264172 gal/L

Worked example — heat 10 gal from 64F to 140F (dT = 76 F):
  1858 Wh  =  1.858 kWh


## 2 · Water demand &amp; tank sizing

Everything downstream scales from **water-equivalent occupancy** — a head count
in which children count as a fraction of an adult:

$$n_{\text{eq}} = n_{\text{adult}} + n_{\text{guest}} + n_{\text{child}}\cdot\frac{f_{\text{child}}}{100}$$

**Per-fixture daily demand** (`fixGal`) — flow × water-on minutes × occupancy (if
the fixture scales per person):

$$V_{\text{fix}} = \dot V_{\text{gpm}} \cdot t_{\min} \cdot \big(n_{\text{eq}}\ \text{if per-person else } 1\big)\quad[\text{gal/day}]$$

**Hot fraction** of a mixed draw (`hotFracAt`) — the share drawn from the hot tank
to hit a mix temperature, clamped to $[0,1]$:

$$f_{\text{hot}} = \operatorname{clamp}\!\left(\frac{T_{\text{mix}}-T_{\text{inlet}}}{T_{\text{set}}-T_{\text{inlet}}},\,0,\,1\right)$$

**Daily totals** — fresh is the sum over all fixtures; hot uses $f_{\text{hot}}$ for
shower-like fixtures and each fixture's `pctHot` otherwise; grey sums only drained
fixtures:

$$V_{\text{fresh}}=\sum_i V_i,\quad
V_{\text{hot}}=\sum_i V_i\,f_{\text{hot},i},\quad
V_{\text{grey}}=\sum_{i\in\text{grey}} V_i$$

**Peak flow** — worst simultaneous indoor draw (two largest indoor fixtures) vs. a
single hose bib:

$$\dot V_{\text{peak}}=\max\!\big(\dot V_{(1)}+\dot V_{(2)},\ \dot V_{\text{hose}}\big)$$

**Autonomy tank sizing** — only autonomy-flagged fixtures, over the storm-gap days
plus a reserve fraction:

$$V_{\text{tank}} = V_{\text{auto}} \cdot d_{\text{water}} \cdot \left(1+\frac{r\%}{100}\right),
\qquad m_{\text{full}} = (V_{\text{fresh,tank}}+V_{\text{grey,tank}})\cdot 8.34\ \text{lb}$$


In [2]:
# ── Water demand & tank sizing ──────────────────────────────────────────────
import math
RHO_W_LB_GAL = 8.34             # lb/gal, density of water

# Occupancy (app default: 2 adults)
ADULTS, CHILDREN, GUESTS = 2, 0, 0
CHILD_FACTOR_PCT = 60          # a child = 60% of an adult's water use

# Temperatures that set the shower hot fraction
INLET_F   = 64
SETPOINT_F = 140
SHOWER_MIX_F = 105

# Autonomy sizing
WATER_DAYS  = 5
RESERVE_PCT = 15

# Fixtures: (name, gpm, min/day, pctHot, grey?, per_person?, auto?, shower_like?)
FIXTURES = [
    ("Kitchen sink",  1.2,  5, 60, True,  True,  True,  False),
    ("Bathroom sink", 1.2,  4, 50, True,  True,  True,  False),
    ("Shower",        1.75, 2, 77, True,  True,  True,  True ),
    ("Hose bib",      3.0,  2,  0, False, False, False, False),
]

occ_eq = ADULTS + GUESTS + CHILDREN * CHILD_FACTOR_PCT / 100
hot_frac = min(1, max(0, (SHOWER_MIX_F - INLET_F) / (SETPOINT_F - INLET_F))) if SETPOINT_F > INLET_F else 1

total_fresh = hot_daily = grey_daily = auto_fresh = auto_grey = 0.0
for name, gpm, mins, pctHot, grey, pp, auto, shower in FIXTURES:
    g = gpm * mins * (occ_eq if pp else 1)          # fixGal
    total_fresh += g
    hp = hot_frac if shower else pctHot / 100
    hot_daily += g * hp
    if grey: grey_daily += g
    if auto: auto_fresh += g
    if auto and grey: auto_grey += g

# Peak flow: two largest indoor (grey) fixtures vs. hose bib
indoor = sorted((f[1] for f in FIXTURES if f[4]), reverse=True)     # grey == indoor
peak_indoor = (indoor[0] if len(indoor) > 0 else 0) + (indoor[1] if len(indoor) > 1 else 0)
hose = max((f[1] for f in FIXTURES if not f[4]), default=0)
peak_flow = max(peak_indoor, hose)

resv = 1 + RESERVE_PCT / 100
fresh_tank = auto_fresh * WATER_DAYS * resv
grey_tank  = auto_grey  * WATER_DAYS * resv
tank_mass_lb = (fresh_tank + grey_tank) * RHO_W_LB_GAL

print("Water demand & tanks (defaults)")
print(f"  Water-equivalent occupancy : {occ_eq:.1f}")
print(f"  Shower hot fraction        : {hot_frac:.3f}  (mix {SHOWER_MIX_F}F of {INLET_F}->{SETPOINT_F}F)")
print(f"  Total fresh                : {total_fresh:.1f} gal/day")
print(f"  Hot water                  : {hot_daily:.1f} gal/day")
print(f"  Grey produced              : {grey_daily:.1f} gal/day")
print(f"  Peak flow                  : {peak_flow:.2f} GPM  ({'2 fixtures' if peak_indoor>=hose else 'hose bib'})")
print(f"  Fresh tank                 : {fresh_tank:.0f} gal  ({WATER_DAYS}d + {RESERVE_PCT}%, autonomy fixtures only)")
print(f"  Grey tank                  : {grey_tank:.0f} gal")
print(f"  Full tanks water mass      : {tank_mass_lb:.0f} lb")

Water demand & tanks (defaults)
  Water-equivalent occupancy : 2.0
  Shower hot fraction        : 0.539  (mix 105F of 64->140F)
  Total fresh                : 34.6 gal/day
  Hot water                  : 15.8 gal/day
  Grey produced              : 28.6 gal/day
  Peak flow                  : 3.00 GPM  (hose bib)
  Fresh tank                 : 164 gal  (5d + 15%, autonomy fixtures only)
  Grey tank                  : 164 gal
  Full tanks water mass      : 2743 lb


## 3 · Hot water — reheat energy, recovery time &amp; standby loss

**Daily reheat energy** for the hot draws, and the **full cold-start** energy to
bring the whole tank from inlet to setpoint:

$$E_{\text{daily}}=\text{heatWh}(V_{\text{hot}},\,\Delta T),\qquad
E_{\text{tank}}=\text{heatWh}(V_{\text{tank}},\,\Delta T),\qquad \Delta T=T_{\text{set}}-T_{\text{inlet}}$$

**Recovery time** — how long the resistance element takes to reheat a full tank:

$$t_{\text{recovery}}=\frac{E_{\text{tank}}}{P_{\text{element}}}\cdot 60\ \text{min}$$

**Standby loss** — steady conduction through the tank insulation, treated as a
plane wall of area $A$ (ft²), thermal resistance $R=R_{/\text{in}}\cdot x_{\text{ins}}$,
across $\Delta T_{\text{amb}}=T_{\text{set}}-T_{\text{amb}}$:

$$\dot Q_{\text{standby}}=\frac{A\,(T_{\text{set}}-T_{\text{amb}})}{R}\cdot\frac{1}{3.412}\ \text{W},
\qquad E_{\text{standby}}=24\,\dot Q_{\text{standby}}\ \text{Wh/day}$$

**Element current** — $I = P_{\text{element}}/V_{\text{bus}}$.

Tank surface area comes from the chosen tank geometry (§5).


In [3]:
# ── Hot water: reheat energy, recovery, standby loss ────────────────────────
import math

RHO_W_LB_GAL, BTU_PER_WH, IN3_PER_GAL = 8.34, 3.412, 231
def heat_wh(gal, dT): return RHO_W_LB_GAL * gal * dT / BTU_PER_WH

# Inputs (defaults)
INLET_F, SETPOINT_F = 64, 140
TANK_GAL   = 10
ELEMENT_W  = 1500
INSUL_R_PER_IN = 6.5      # closed-cell foam ~R-6.5/in
INSUL_IN   = 3
AMBIENT_F  = 68
BUS_V      = 48
HOT_DAILY_GAL = 15.78     # from §2 (hot water/day)
TANK_SHAPE = "cyl"        # cube | slab | cyl

def hw_area_ft2(shape, gal):
    # Tank surface area [ft^2] for the app's three tank geometries (hwGeom)
    Vw = gal * IN3_PER_GAL                       # in^3
    if shape == "cube":
        a = Vw ** (1/3);        sa = 6*a*a
    elif shape == "slab":
        t = 8; a = math.sqrt(Vw/t); sa = 2*a*a + 4*a*t
    else:  # cylinder, height = 3 * radius
        r = (Vw/(3*math.pi)) ** (1/3); h = 3*r
        sa = 2*math.pi*r*r + 2*math.pi*r*h
    return sa / 144                              # in^2 -> ft^2

dT = SETPOINT_F - INLET_F
heat_energy_wh = heat_wh(HOT_DAILY_GAL, dT)
tank_energy_wh = heat_wh(TANK_GAL, dT)
recovery_min   = tank_energy_wh / ELEMENT_W * 60
area_ft2 = hw_area_ft2(TANK_SHAPE, TANK_GAL)
R_val    = INSUL_R_PER_IN * INSUL_IN
standby_w = area_ft2 * (SETPOINT_F - AMBIENT_F) / R_val / 3.412
standby_wh_day = standby_w * 24
element_amps = ELEMENT_W / BUS_V

print("Hot water — energy & losses (defaults)")
print(f"  dT (inlet->setpoint)   : {dT} F")
print(f"  Daily reheat energy    : {heat_energy_wh:.0f} Wh/day  (heat {HOT_DAILY_GAL} gal hot draws)")
print(f"  Full cold-start energy : {tank_energy_wh:.0f} Wh  ->  recovery {recovery_min:.0f} min @ {ELEMENT_W} W")
print(f"  Tank surface area      : {area_ft2:.2f} ft2  ({TANK_SHAPE})")
print(f"  Insulation R-value     : R-{R_val:.0f}  ({INSUL_R_PER_IN}/in x {INSUL_IN} in)")
print(f"  Standby loss           : {standby_w:.1f} W  ->  {standby_wh_day:.0f} Wh/day (24/7)")
print(f"  Element current        : {element_amps:.1f} A DC ({ELEMENT_W} W / {BUS_V} V)")

Hot water — energy & losses (defaults)
  dT (inlet->setpoint)   : 76 F
  Daily reheat energy    : 2931 Wh/day  (heat 15.78 gal hot draws)
  Full cold-start energy : 1858 Wh  ->  recovery 74 min @ 1500 W
  Tank surface area      : 6.84 ft2  (cyl)
  Insulation R-value     : R-20  (6.5/in x 3 in)
  Standby loss           : 7.4 W  ->  178 Wh/day (24/7)
  Element current        : 31.2 A DC (1500 W / 48 V)


## 4 · Hot water — shower-session capacity &amp; tank geometry

A **military (navy) shower** session: mixed water per session scales with occupancy
and a **factor of ignorance** (safety margin); the hot portion is drawn from the
tank at $f_{\text{hot}}$:

$$V_{\text{mix}}=\dot V_{\text{sh}}\,t_{\text{sh}}\,n_{\text{eq}}\,\text{FoI},\qquad
V_{\text{hot,session}}=V_{\text{mix}}\,f_{\text{hot}}$$

**Consecutive showers per tank** — how many single showers a full tank supports
before the hot runs out:

$$V_{\text{hot,person}}=\dot V_{\text{sh}}\,t_{\text{sh}}\,\text{FoI}\,f_{\text{hot}},\qquad
N_{\text{showers}}=\frac{V_{\text{tank}}}{V_{\text{hot,person}}}$$

**Tank geometry** (`hwGeom`) — surface area (drives standby) and bare dimensions
for the three shapes, with water volume $V_w=231\,V_{\text{gal}}$ in³:

| Shape | Constraint | Surface area |
|---|---|---|
| Cube | $a=\sqrt[3]{V_w}$ | $6a^2$ |
| Low-profile slab | thickness $t=8''$, $a=\sqrt{V_w/t}$ | $2a^2+4at$ |
| Cylinder | $h=3r,\ r=\sqrt[3]{V_w/3\pi}$ | $2\pi r^2+2\pi r h$ |


In [4]:
# ── Hot water: shower session capacity & tank geometry ──────────────────────
import math
IN3_PER_GAL = 231

# Inputs (defaults)
SHOWER_GPM, SHOWER_MIN = 1.75, 2
FOI = 1.5                 # factor of ignorance (safety margin on session volume)
OCC_EQ = 2.0
INLET_F, SETPOINT_F, SHOWER_MIX_F = 64, 140, 105
TANK_GAL = 10

hot_frac = min(1, max(0, (SHOWER_MIX_F - INLET_F)/(SETPOINT_F - INLET_F)))
mix_gal        = SHOWER_GPM * SHOWER_MIN * OCC_EQ * FOI
hot_need       = mix_gal * hot_frac
hot_per_person = SHOWER_GPM * SHOWER_MIN * FOI * hot_frac
consec_showers = TANK_GAL / hot_per_person if hot_per_person > 0 else float('inf')

def hw_geom(shape, gal):
    Vw = gal * IN3_PER_GAL
    if shape == "cube":
        a = Vw ** (1/3);  return 6*a*a, f"{a:.1f} in cube"
    if shape == "slab":
        t = 8; a = math.sqrt(Vw/t); return 2*a*a+4*a*t, f"{a:.1f}x{a:.1f}x{t:.0f} in"
    r = (Vw/(3*math.pi)) ** (1/3); h = 3*r
    return 2*math.pi*r*r + 2*math.pi*r*h, f"{2*r:.1f} in dia x {h:.1f} in"

print("Shower session capacity (defaults)")
print(f"  Hot fraction           : {hot_frac:.3f}")
print(f"  Mixed water / session  : {mix_gal:.1f} gal  ({SHOWER_GPM} GPM x {SHOWER_MIN} min x {OCC_EQ} occ x FoI {FOI})")
print(f"  Hot drawn / session    : {hot_need:.1f} gal")
print(f"  Hot / single shower    : {hot_per_person:.2f} gal")
print(f"  Consecutive showers/tank: {consec_showers:.1f}  (tank {TANK_GAL} gal)")
print()
print("Tank geometry — surface area by shape (10 gal):")
for shp in ("cube", "slab", "cyl"):
    sa_in2, dims = hw_geom(shp, TANK_GAL)
    print(f"  {shp:5s}: {sa_in2/144:.2f} ft2   ({dims})")

Shower session capacity (defaults)
  Hot fraction           : 0.539
  Mixed water / session  : 10.5 gal  (1.75 GPM x 2 min x 2.0 occ x FoI 1.5)
  Hot drawn / session    : 5.7 gal
  Hot / single shower    : 2.83 gal
  Consecutive showers/tank: 3.5  (tank 10 gal)

Tank geometry — surface area by shape (10 gal):
  cube : 7.28 ft2   (13.2 in cube)
  slab : 7.79 ft2   (17.0x17.0x8 in)
  cyl  : 6.84 ft2   (12.5 in dia x 18.8 in)


## 5 · Grey drainage (DWV) — fixture units &amp; pipe sizing

Drainage is sized by **Drainage Fixture Units (DFU)** — a dimensionless load each
fixture places on the waste system. Only drained (grey) fixtures count; the hose
bib and composting toilet contribute nothing.

$$\text{DFU}_{\text{total}}=\sum_{i\in\text{grey}}\text{DFU}_i$$

**Branch (individual trap arm)** size from each fixture's DFU, and the **main grey
line** from the total — step tables per HUD 24 CFR §3280.610–611:

| Branch DFU | Min branch | | Total DFU | Main grey line |
|---|---|---|---|---|
| ≤ 1 | 1¼″ | | ≤ 6 | 2″ |
| ≤ 2 | 1½″ | | ≤ 12 | 3″ |
| > 2 | 2″ | | > 12 | 4″ |

Horizontals graded **¼″ per foot** (§3280.610); minimum **1½″ vents** (§3280.611).


In [5]:
# ── Grey drainage (DWV): DFU totals -> pipe sizing ──────────────────────────
# (name, dfu, trap_in, grey?)
DRAIN_FIXTURES = [
    ("Kitchen sink",  2, 1.5,  True),
    ("Bathroom sink", 1, 1.25, True),
    ("Shower",        2, 2.0,  True),
    ("Hose bib",      0, 0.0,  False),   # not drained -> excluded
]

def branch_size(dfu):
    return "1-1/4\"" if dfu <= 1 else "1-1/2\"" if dfu <= 2 else "2\""

grey = [f for f in DRAIN_FIXTURES if f[3]]
total_dfu = sum(f[1] for f in grey)
main = "2\"" if total_dfu <= 6 else "3\"" if total_dfu <= 12 else "4\""

print("Grey drainage (DWV)")
print(f"  {'Fixture':16s} {'Trap':6s} {'DFU':>4s}  Min branch")
for name, dfu, trap, _ in grey:
    print(f"  {name:16s} {trap:>4}\"  {dfu:>4d}  {branch_size(dfu)}")
print(f"  {'TOTAL':16s} {'':6s} {total_dfu:>4d}")
print()
print(f"  Main grey drain : {main}   (<= {6 if total_dfu<=6 else 12 if total_dfu<=12 else '-'} DFU)")
print(f"  Min vent        : {'1-1/2\"' if total_dfu>0 else '-'}   (HUD 3280.611)")
print(f"  Grade           : 1/4\" per foot on horizontals (3280.610)")

Grey drainage (DWV)
  Fixture          Trap    DFU  Min branch
  Kitchen sink      1.5"     2  1-1/2"
  Bathroom sink    1.25"     1  1-1/4"
  Shower            2.0"     2  1-1/2"
  TOTAL                      5

  Main grey drain : 2"   (<= 6 DFU)
  Min vent        : 1-1/2"   (HUD 3280.611)
  Grade           : 1/4" per foot on horizontals (3280.610)


## 6 · Composting toilet — liquid handling

A dry composting toilet uses **no flush water**. The only liquid to manage is
urine. Two modes:

- **Urine separation** — all urine diverted at the seat to a container.
- **Leachate tank** — only a fraction $f_{\text{leach}}$ of urine drains off (the
  rest is absorbed/evaporated in the compost).

$$V_{\text{urine}}=n_{\text{eq}}\cdot u_{\text{Lppd}}\cdot\text{L2GAL}\ \ [\text{gal/day}]$$

$$V_{\text{liquid}}=\begin{cases}V_{\text{urine}} & \text{separation}\\[2pt]
V_{\text{urine}}\cdot\dfrac{f_{\text{leach}}\%}{100} & \text{leachate}\end{cases}
\qquad
V_{\text{container}}=V_{\text{liquid}}\cdot d_{\text{empty}},\quad
\text{empties/mo}=\frac{30}{d_{\text{empty}}}$$


In [6]:
# ── Composting toilet: urine / leachate volume -> container sizing ──────────
L2GAL = 0.264172

OCC_EQ = 2.0
URINE_LPPD = 1.4          # litres urine per person per day
EMPTY_DAYS = 7
LEACHATE_FRAC_PCT = 35
MODE = "urine-sep"        # 'urine-sep' | 'leachate'

urine_gal_day = OCC_EQ * URINE_LPPD * L2GAL
liquid_gal_day = urine_gal_day if MODE == "urine-sep" else urine_gal_day * LEACHATE_FRAC_PCT/100
container_gal  = liquid_gal_day * EMPTY_DAYS
empties_per_mo = 30 / EMPTY_DAYS
label = "Urine container" if MODE == "urine-sep" else "Leachate tank"

print("Composting toilet (defaults)")
print(f"  Mode                : {MODE}")
print(f"  Urine produced      : {urine_gal_day:.2f} gal/day  ({OCC_EQ} occ x {URINE_LPPD} L)")
print(f"  Liquid to manage    : {liquid_gal_day:.2f} gal/day")
print(f"  {label:19s} : {container_gal:.1f} gal  ({EMPTY_DAYS}d between empties)")
print(f"  Empties             : {empties_per_mo:.1f} / month")
print(f"  Flush water         : 0 gal/day (composting)")

Composting toilet (defaults)
  Mode                : urine-sep
  Urine produced      : 0.74 gal/day  (2.0 occ x 1.4 L)
  Liquid to manage    : 0.74 gal/day
  Urine container     : 5.2 gal  (7d between empties)
  Empties             : 4.3 / month
  Flush water         : 0 gal/day (composting)


## 7 · Pump sizing — friction, Total Dynamic Head &amp; power

The pump is sized around the **shower critical loop**. The governing fixture sets
the design flow; static lift + fixture residual pressure + friction set the head.

**Design (duty) flow** — shower flow inflated by a simultaneity factor:

$$Q_{\text{duty}}=\dot V_{\text{sh}}\cdot F_{\text{simul}}$$

**Hazen–Williams friction** for PEX ($C=150$) — head-loss gradient (ft head per ft
of pipe) at flow $Q$ (GPM) through inside diameter $d$ (in):

$$S(Q,d)=0.002083\left(\frac{100}{C}\right)^{1.852}\frac{Q^{1.852}}{d^{4.8655}}$$

Fittings are added as **equivalent length** on the ½″ branch (each 90° elbow ≈ 1.5 ft,
each tee ≈ 3 ft):

$$L_{\text{branch,eq}}=L_{\text{branch}}+1.5\,n_{90}+3\,n_{\text{tee}}$$

$$h_f(Q)=S(Q,d_{\text{header}})\,L_{\text{header}}+S(Q,d_{½})\,L_{\text{branch,eq}}$$

**Total Dynamic Head** — the duty point the pump must hit:

$$\text{TDH}(Q)=h_{\text{static}}+h_{\text{fixture}}+h_f(Q),\qquad
P_{\text{req}}=\frac{\text{TDH}}{2.3067}\ \text{psi}$$

**Velocity check** (keep ≤ 5 ft/s): $\;v=0.4085\,Q/d^{2}$.

**Power** — hydraulic then electrical (the constant 0.1885 = W per GPM·ft):

$$P_{\text{hyd}}=0.1885\,Q_{\text{duty}}\,\text{TDH},\qquad
P_{\text{elec}}=\frac{P_{\text{hyd}}}{\eta_{\text{pump}}},\qquad
I=\frac{P_{\text{elec}}}{V_{\text{bus}}}$$

**Operating point** — intersection of the pump curve $H_{\text{pump}}(Q)$ (piecewise-linear)
and the system curve $\text{TDH}(Q)$; if $H_{\text{pump}}(0)<\text{TDH}(0)$ the pump can't
overcome static head and there is no flow.


In [7]:
# ── Pump sizing: Hazen-Williams friction, TDH, velocity, power ──────────────
import math

# Pipe inside diameters (in) and pump/loop inputs (defaults)
PEX_HALF_ID, PEX_3Q_ID = 0.485, 0.681
HEADER_SIZE = "3/4"            # '3/4' -> 0.681 ID, else 1/2 -> 0.485
HEADER_LEN_FT = 10
BRANCH_LEN_FT = 8
N_90, N_TEE = 4, 2
SHOWER_GPM = 1.75
SIMUL_FACTOR = 1.3
STATIC_LIFT_FT = 7.22
FIXTURE_HEAD_FT = 46          # residual pressure required at the shower, as head
PUMP_EFF_PCT = 40
BUS_V = 48
MAX_SYS_PSI = 80
PSI2FT = 2.3067
TOTAL_FRESH_GAL = 34.6        # from §2 (for daily pump runtime/energy)

# Pump candidate curve: list of (Q GPM, H ft)
PUMP_CURVE = [(0,127),(1,118),(2,106),(3,92),(4,60),(5,20)]

def hw_slope(q, d, C=150):
    if q <= 0: return 0.0
    return 0.002083 * (100/C)**1.852 * q**1.852 / d**4.8655
def pipe_vel(q, d):  return 0.4085 * q / (d*d)          # ft/s
def header_id():     return PEX_3Q_ID if HEADER_SIZE == "3/4" else PEX_HALF_ID
def branch_equiv_ft(): return BRANCH_LEN_FT + N_90*1.5 + N_TEE*3
def loop_friction_ft(q):
    return hw_slope(q, header_id())*HEADER_LEN_FT + hw_slope(q, PEX_HALF_ID)*branch_equiv_ft()
def sys_head(q):     return STATIC_LIFT_FT + FIXTURE_HEAD_FT + loop_friction_ft(q)   # TDH

def pump_head(q):    # piecewise-linear interpolation / clamp on the curve
    c = sorted(PUMP_CURVE)
    if q <= c[0][0]:  return c[0][1]
    if q >= c[-1][0]: return c[-1][1]
    for (q0,h0),(q1,h1) in zip(c, c[1:]):
        if q0 <= q <= q1:
            return h0 + (h1-h0)*(q-q0)/(q1-q0)
    return 0.0

def operating_point():   # walk flow up until pump curve drops below system curve
    qmax = max(q for q,_ in PUMP_CURVE)
    if pump_head(0) < sys_head(0): return 0.0, sys_head(0), True   # dead-headed
    prev = pump_head(0) - sys_head(0); x = 0.02
    while x <= qmax:
        d = pump_head(x) - sys_head(x)
        if prev >= 0 and d < 0:
            t = prev/(prev-d); q = (x-0.02) + t*0.02
            return q, sys_head(q), False
        prev = d; x += 0.02
    return qmax, sys_head(qmax), False

# Duty point
duty_flow = SHOWER_GPM * SIMUL_FACTOR
header_fric = hw_slope(duty_flow, header_id()) * HEADER_LEN_FT
branch_fric = hw_slope(duty_flow, PEX_HALF_ID) * branch_equiv_ft()
tdh = sys_head(duty_flow)
req_psi = tdh / PSI2FT
v_header = pipe_vel(duty_flow, header_id())
hydraulic_w = duty_flow * tdh * 0.1885
pump_elec_w = hydraulic_w / max(PUMP_EFF_PCT/100, 0.01)
pump_amps = pump_elec_w / BUS_V

op_q, op_h, dead = operating_point()
provide_head = pump_head(duty_flow)
can_meet = provide_head >= tdh and not dead
runtime_min = (TOTAL_FRESH_GAL / max(op_q, 0.01)) if not dead else float('inf')
pump_wh_day = 0 if dead else pump_elec_w * runtime_min/60

print("Pump sizing (defaults, 3/4\" header + 1/2\" branch, Hazen-Williams C=150)")
print(f"  Design (duty) flow  : {duty_flow:.2f} GPM   (shower {SHOWER_GPM} x {SIMUL_FACTOR})")
print(f"  Friction: header {header_fric:.2f} ft + branch {branch_fric:.2f} ft = {header_fric+branch_fric:.2f} ft")
print(f"  Total Dynamic Head  : {tdh:.1f} ft  =  {req_psi:.0f} psi  (lift {STATIC_LIFT_FT} + residual {FIXTURE_HEAD_FT} + friction)")
print(f"  Header velocity     : {v_header:.2f} ft/s  ({'OK <=5' if v_header<=5 else 'HIGH >5'})")
print(f"  Hydraulic power     : {hydraulic_w:.1f} W   (0.1885 x Q x TDH)")
print(f"  Electrical power    : {pump_elec_w:.0f} W  ->  {pump_amps:.2f} A DC  (hydraulic / {PUMP_EFF_PCT}% eff)")
print(f"  Pump at duty        : {provide_head:.0f} ft  ({'meets' if can_meet else 'SHORT of'} {tdh:.0f} ft)")
print(f"  Operating point     : {op_q:.2f} GPM @ {op_h:.0f} ft" + ("  (dead-headed)" if dead else ""))
print(f"  Pressure vs cap      : {req_psi:.0f} psi vs {MAX_SYS_PSI} psi max ({'OK' if req_psi<=MAX_SYS_PSI else 'OVER'})")
print(f"  Pump energy         : {pump_wh_day:.0f} Wh/day  ({runtime_min:.0f} min/day at {op_q:.2f} GPM)")

Pump sizing (defaults, 3/4" header + 1/2" branch, Hazen-Williams C=150)
  Design (duty) flow  : 2.27 GPM   (shower 1.75 x 1.3)
  Friction: header 0.29 ft + branch 3.05 ft = 3.34 ft
  Total Dynamic Head  : 56.6 ft  =  25 psi  (lift 7.22 + residual 46 + friction)
  Header velocity     : 2.00 ft/s  (OK <=5)
  Hydraulic power     : 24.3 W   (0.1885 x Q x TDH)
  Electrical power    : 61 W  ->  1.26 A DC  (hydraulic / 40% eff)
  Pump at duty        : 102 ft  (meets 57 ft)
  Operating point     : 3.93 GPM @ 62 ft
  Pressure vs cap      : 25 psi vs 80 psi max (OK)
  Pump energy         : 9 Wh/day  (9 min/day at 3.93 GPM)


## 8 · DC power balance — water system vs budget

The water system's daily electrical energy is the sum of the three loads computed
above, checked against a daily budget:

$$E_{\text{water}}=E_{\text{hot}}+E_{\text{standby}}+E_{\text{pump}},\qquad
\text{used}\%=\frac{E_{\text{water}}}{E_{\text{budget}}}\cdot100$$

**Peak current** — the heater element and pump running together:

$$I_{\text{peak}}=I_{\text{element}}+I_{\text{pump}},\qquad
\text{headroom}=E_{\text{budget}}-E_{\text{water}}$$


In [8]:
# ── DC power balance: water-system energy vs budget ─────────────────────────
# Daily energies from sections 3 & 7 (defaults)
HEAT_ENERGY_WH  = 2931       # daily hot-draw reheat  (§3)
STANDBY_WH_DAY  = 178        # 24/7 tank standby      (§3)
PUMP_WH_DAY     = 122        # pumping                (§7, at operating point)
BUDGET_WH       = 4500
ELEMENT_AMPS    = 1500/48    # §3
PUMP_AMPS       = 1.26       # §7

water_wh_day = HEAT_ENERGY_WH + STANDBY_WH_DAY + PUMP_WH_DAY
balance_pct  = water_wh_day / BUDGET_WH * 100
peak_amps    = ELEMENT_AMPS + PUMP_AMPS
headroom     = BUDGET_WH - water_wh_day

print("DC power balance — water system")
print(f"  Hot-water heating   : {HEAT_ENERGY_WH:>5d} Wh/day")
print(f"  Standby loss        : {STANDBY_WH_DAY:>5d} Wh/day")
print(f"  Pumping             : {PUMP_WH_DAY:>5d} Wh/day")
print(f"  Water total         : {water_wh_day:>5.0f} Wh/day  of {BUDGET_WH} budget ({balance_pct:.0f}%)")
print(f"  Headroom            : {headroom:>5.0f} Wh/day")
print(f"  Peak current        : {peak_amps:.1f} A DC (heater {ELEMENT_AMPS:.1f} + pump {PUMP_AMPS:.1f})")

DC power balance — water system
  Hot-water heating   :  2931 Wh/day
  Standby loss        :   178 Wh/day
  Pumping             :   122 Wh/day
  Water total         :  3231 Wh/day  of 4500 budget (72%)
  Headroom            :  1269 Wh/day
  Peak current        : 32.5 A DC (heater 31.2 + pump 1.3)


## 9 · Battery pack &amp; autonomy

The pack is **defined** (modules × spec) and autonomy is the *output*.

$$E_{\text{installed}}=n_{\text{mod}}\,E_{\text{mod}},\qquad
E_{\text{usable}}=E_{\text{installed}}\cdot\frac{\text{DoD}\%}{100}\cdot\frac{\eta_{\text{sys}}\%}{100}$$

**Autonomy** achieved vs. the required bank for a target number of no-sun days:

$$d_{\text{achieved}}=\frac{E_{\text{usable}}}{E_{\text{load,total}}},\qquad
E_{\text{req}}=\frac{E_{\text{load,total}}\cdot d_{\text{power}}}{\frac{\text{DoD}\%}{100}\cdot\frac{\eta_{\text{sys}}\%}{100}},\qquad
n_{\text{req}}=\left\lceil\frac{E_{\text{req}}}{E_{\text{mod}}}\right\rceil$$

Total load $E_{\text{load,total}}=E_{\text{water}}+E_{\text{house}}$, where
$E_{\text{house}}=\sum(\text{W}\times\text{h/day})$ over the electrical-loads table
(including the computed HVAC row).


In [9]:
# ── Battery pack & autonomy ─────────────────────────────────────────────────
import math

# Pack: 2x Pytes V5 (LFP, 5.12 kWh / 100 Ah each)
BATT_MODULES  = 2
BATT_MOD_KWH  = 5.12
BATT_MOD_AH   = 100
DOD_PCT       = 90
SYS_EFF_PCT   = 95
POWER_DAYS    = 3

WATER_WH_DAY  = 3231         # §8
HOUSE_WH_DAY  = 2497         # sum of loads table (W x h/day), incl. computed HVAC

total_load_wh = WATER_WH_DAY + HOUSE_WH_DAY
mod_wh        = BATT_MOD_KWH * 1000
installed_wh  = BATT_MODULES * mod_wh
installed_ah  = BATT_MODULES * BATT_MOD_AH
usable_wh     = installed_wh * (DOD_PCT/100) * (SYS_EFF_PCT/100)
days_achieved = usable_wh / total_load_wh if total_load_wh else float('inf')
req_bank_wh   = total_load_wh * POWER_DAYS / ((DOD_PCT/100)*(SYS_EFF_PCT/100))
req_modules   = max(1, math.ceil(req_bank_wh / mod_wh))
meets = days_achieved >= POWER_DAYS

print("Battery pack & autonomy")
print(f"  Total daily load    : {total_load_wh:.0f} Wh/day (water {WATER_WH_DAY} + house {HOUSE_WH_DAY})")
print(f"  Installed pack      : {installed_wh/1000:.2f} kWh  ({BATT_MODULES}x V5, {installed_ah:.0f} Ah)")
print(f"  Usable energy       : {usable_wh/1000:.2f} kWh  ({DOD_PCT}% DoD x {SYS_EFF_PCT}% eff)")
print(f"  Autonomy achieved   : {days_achieved:.1f} days  (target {POWER_DAYS})")
print(f"  Modules for target  : {req_modules}  ({'OK, have '+str(BATT_MODULES) if meets else 'need '+str(req_modules)+', have '+str(BATT_MODULES)})")

Battery pack & autonomy
  Total daily load    : 5728 Wh/day (water 3231 + house 2497)
  Installed pack      : 10.24 kWh  (2x V5, 200 Ah)
  Usable energy       : 8.76 kWh  (90% DoD x 95% eff)
  Autonomy achieved   : 1.5 days  (target 3)
  Modules for target  : 4  (need 4, have 2)


## 10 · Solar generation &amp; daily profile

**Array output** — nameplate STC watts (with any bifacial gain), harvested over
peak-sun-hours and knocked down by a system derate:

$$P_{\text{array}}=P_{\text{panel}}\cdot n_{\text{panel}}\cdot\left(1+\frac{\text{bifacial}\%}{100}\right)$$

$$E_{\text{solar}}=P_{\text{array}}\cdot\text{PSH}\cdot\frac{\text{derate}\%}{100},\qquad
\text{cover}\%=\frac{E_{\text{solar}}}{E_{\text{load,total}}}\cdot100,\qquad
E_{\text{net}}=E_{\text{solar}}-E_{\text{load,total}}$$

**Daily generation shape** — a raised half-sine over the daylight window,
normalized so the 24-hour area sums to 1 (used to stack solar against hourly load):

$$a_h=\sin\!\left(\pi\,\frac{c_h-t_{\text{start}}}{d_{\text{light}}}\right)\ \text{for }t_{\text{start}}\le c_h\le t_{\text{end}},\qquad
\hat a_h=\frac{a_h}{\sum_j a_j}$$

with $c_h=h+0.5$, window centered on noon of width $d_{\text{light}}$.


In [10]:
# ── Solar generation & normalized daily profile ─────────────────────────────
import math

# Array: 3x Hyundai HiN-T595NI (595 W STC), flush-mounted (no bifacial gain)
PANEL_W, PANEL_QTY, BIFACIAL_PCT = 595, 3, 0
SOLAR_PSH    = 6.8            # Austin summer peak-sun-hours
SOLAR_DERATE = 75            # % system derate
DAYLIGHT_H   = 14            # Austin summer daylight window
TOTAL_LOAD_WH = 5728         # §9

solar_w      = PANEL_W * PANEL_QTY * (1 + BIFACIAL_PCT/100)
solar_wh_day = solar_w * SOLAR_PSH * (SOLAR_DERATE/100)
cover_pct    = solar_wh_day / TOTAL_LOAD_WH * 100 if TOTAL_LOAD_WH else 0
net          = solar_wh_day - TOTAL_LOAD_WH

def solar_profile(daylight_h):
    start, end = 12 - daylight_h/2, 12 + daylight_h/2
    a = []
    for h in range(24):
        c = h + 0.5
        a.append(math.sin(math.pi*(c-start)/daylight_h) if start <= c <= end else 0.0)
    s = sum(a) or 1
    return [x/s for x in a]

prof = solar_profile(DAYLIGHT_H)
peak_h = max(range(24), key=lambda h: prof[h])

print("Solar generation (Austin summer defaults)")
print(f"  Array power         : {solar_w:.0f} W  ({PANEL_QTY} x {PANEL_W} W)")
print(f"  Daily harvest       : {solar_wh_day/1000:.2f} kWh/day  ({solar_w:.0f} W x {SOLAR_PSH} h x {SOLAR_DERATE}%)")
print(f"  Load covered        : {cover_pct:.0f}%  of {TOTAL_LOAD_WH/1000:.2f} kWh")
print(f"  Net daily           : {'+' if net>=0 else ''}{net:.0f} Wh/day  ({'surplus' if net>=0 else 'deficit'})")
print(f"  Profile area check  : sum = {sum(prof):.3f}  (normalized to 1); peak hour ~{peak_h}:30")

Solar generation (Austin summer defaults)
  Array power         : 1785 W  (3 x 595 W)
  Daily harvest       : 9.10 kWh/day  (1785 W x 6.8 h x 75%)
  Load covered        : 159%  of 5.73 kWh
  Net daily           : +3376 Wh/day  (surplus)
  Profile area check  : sum = 1.000  (normalized to 1); peak hour ~11:30


## 11 · Thermal envelope — steel-bridged R → UA → U₀

The envelope is a fixed exterior box (144×90×99 in). Each surface's effective
**R-value** accounts for **steel thermal bridging**, then areas roll up to a
conductance $UA$ and an overall coefficient $U_0$ checked against the HUD zone cap.

**Series layers** wrap both thermal paths (air films + sheathing + continuous
insulation + panel). For a wall:

$$R_{\text{series}}=R_{\text{film}}+x_{\text{sheath}}R_{\text{wood}}+x_{\text{cont}}R_{\text{ins}}+x_{\text{panel}}R_{\text{pvc}}$$

**Cavity layer with steel studs** — cavity insulation between steel studs is only
~45–50 % effective; the app interpolates ASHRAE 90.1 App A / IECC Table A103.3.6.2
effective-R by nominal cavity R and stud spacing. Nominal cavity $R=R_{/\text{in}}\cdot x_{\text{stud}}$.

$$R_{\text{surface}}=R_{\text{series}}+R_{\text{cav,eff}}(\text{A103})$$

**HUD basis** instead uses a parallel-path with prescribed framing fractions
(15 % wall, 10 % roof/floor):

$$R_{\text{surface}}=\left[\frac{f\!f}{R_{\text{series}}+R_{\text{steel}}}+\frac{1-f\!f}{R_{\text{series}}+R_{\text{cav}}}\right]^{-1}$$

**Roll-up** over the six surface groups (opaque wall, roof, floor, window, door):

$$UA=\sum_i\frac{A_i}{R_i},\qquad U_0=\frac{UA}{A_{\text{env}}},\qquad
A_{\text{env}}=A_{\text{wall}}+A_{\text{roof}}+A_{\text{floor}}$$

Pass if $U_0\le U_{0,\text{zone}}$ where the HUD §3280.506 caps are Zone 1 = 0.116,
Zone 2 = 0.096, Zone 3 = 0.079 Btu/hr·ft²·°F.


In [11]:
# ── Thermal envelope: steel-bridged assembly R -> UA -> U0 ──────────────────
import math

# Exterior box (in) and material R-values
ENV_L, ENV_W, ENV_H = 144, 90, 99
INS_RPI = {"none":0.0,"batt":3.5,"mineral":4.2,"eps":4.0,"xps":5.0,"polyiso":6.0,"ccspf":6.5}
FILM = {"wall":0.85, "roof":0.78, "floor":1.00}       # air films
MISC_RPI = {"wood":1.25, "pvc":0.90}
UO_ZONE = {1:0.116, 2:0.096, 3:0.079}                 # HUD 3280.506 caps

# A103 effective-R tables: [nominal cavity R, effective R] at 16" and 24" o.c.
A103_16 = [[0,0],[11,5.5],[13,6.0],[15,6.4],[19,7.1],[21,7.4],[25,7.8]]
A103_24 = [[0,0],[11,6.6],[13,7.2],[15,7.8],[19,8.6],[21,9.0],[25,9.6]]

def interp(pts, x):
    if x <= pts[0][0]:
        a,b = pts[0],pts[1]; return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])
    for a,b in zip(pts, pts[1:]):
        if x <= b[0]: return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])
    a,b = pts[-2],pts[-1]; return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])

def framed_eff_r(nom_cav_r, spacing_in):
    e16, e24 = interp(A103_16, nom_cav_r), interp(A103_24, nom_cav_r)
    s = min(24, max(16, spacing_in))
    eff = e16 + (e24-e16)*((s-16)/8)
    return max(0, min(eff, nom_cav_r or 0.9))

# Design inputs (defaults): polyiso cavity in 3" steel studs @16", no continuous
STUD_DEPTH_IN, STUD_SPACING_IN = 3, 16
SHEATH_IN, PANEL_IN = 0.5, 0.5
WALL_CONT_IN = 0
WINDOW_AREA_FT2, WINDOW_U = 16, 0.30
DOOR_W_IN, DOOR_H_IN, DOOR_R = 32, 80, 6
UO_TARGET_ZONE = 2

# Geometry (ft, ft^2)
L, Wd, H = ENV_L/12, ENV_W/12, ENV_H/12
wall_gross = 2*(L+Wd)*H; roof_a = L*Wd; floor_a = L*Wd
win_a = WINDOW_AREA_FT2; door_a = (DOOR_W_IN*DOOR_H_IN)/144
opaque_wall = max(1, wall_gross - win_a - door_a)
A_env = wall_gross + roof_a + floor_a

# Per-surface effective R (design/spacing basis)
wall_series = FILM["wall"] + SHEATH_IN*MISC_RPI["wood"] + WALL_CONT_IN*INS_RPI["polyiso"] + PANEL_IN*MISC_RPI["pvc"]
R_wall  = wall_series + framed_eff_r(INS_RPI["polyiso"]*STUD_DEPTH_IN, STUD_SPACING_IN)
roof_series  = FILM["roof"] + 0.75*MISC_RPI["wood"] + 0*INS_RPI["polyiso"] + 0.5*MISC_RPI["pvc"]
R_roof  = roof_series + framed_eff_r(INS_RPI["polyiso"]*3, STUD_SPACING_IN)
floor_series = FILM["floor"] + 0.75*MISC_RPI["wood"] + 0*INS_RPI["polyiso"] + 0.5*MISC_RPI["wood"]
R_floor = floor_series + framed_eff_r(INS_RPI["polyiso"]*3, STUD_SPACING_IN)
R_win, R_door = 1/max(WINDOW_U, 0.05), max(DOOR_R, 0.5)

UA = opaque_wall/R_wall + roof_a/R_roof + floor_a/R_floor + win_a/R_win + door_a/R_door
Uo = UA / A_env
cap = UO_ZONE[UO_TARGET_ZONE]

print("Thermal envelope (design basis, polyiso cavity in 3\" steel studs @16\" o.c.)")
print(f"  Areas: wall {opaque_wall:.0f} (opaque) + roof {roof_a:.0f} + floor {floor_a:.0f} + win {win_a:.0f} + door {door_a:.1f} ft2")
print(f"  Effective R-values:")
print(f"    Wall  R-{R_wall:4.1f}   Roof  R-{R_roof:4.1f}   Floor R-{R_floor:4.1f}")
print(f"    Window R-{R_win:4.1f}   Door  R-{R_door:4.1f}")
print(f"  UA (conductance)    : {UA:.1f} Btu/hr-F")
print(f"  U0 (overall)        : {Uo:.4f} Btu/hr-ft2-F")
print(f"  HUD Zone {UO_TARGET_ZONE} cap      : {cap} -> {'PASS' if Uo <= cap else 'FAIL'}")

Thermal envelope (design basis, polyiso cavity in 3" steel studs @16" o.c.)
  Areas: wall 288 (opaque) + roof 90 + floor 90 + win 16 + door 17.8 ft2
  Effective R-values:
    Wall  R- 8.8   Roof  R- 9.1   Floor R- 9.5
    Window R- 3.3   Door  R- 6.0
  UA (conductance)    : 59.7 Btu/hr-F
  U0 (overall)        : 0.1190 Btu/hr-ft2-F
  HUD Zone 2 cap      : 0.096 -> FAIL


## 12 · Thermal loads — heating &amp; cooling design loads

On the real envelope ($UA$, per-surface R from §11), the design loads.

**Heating** — conduction across every surface plus infiltration, at
$\Delta T_h = T_{\text{in}}-T_{\text{out}}$. The certified number takes **no**
internal-gain credit (§3280.510); the operating load credits people + plug heat:

$$Q_{\text{cond}}=\Delta T_h\sum_i\frac{A_i}{R_i},\qquad
Q_{\text{inf}}=1.08\left(\frac{\text{ACH}\cdot V_{\text{ft}^3}}{60}\right)\Delta T_h$$

$$Q_{\text{heat,cert}}=Q_{\text{cond}}+Q_{\text{inf}},\qquad
Q_{\text{heat,oper}}=\max\!\big(0,\ Q_{\text{heat,cert}}-Q_{\text{gains}}\big)$$

(HUD-basis infiltration instead uses $0.7\,(70-T_{\text{out}})\,P_{\text{perim}}$.
The constant $1.08 = 0.018\ \text{Btu/ft}^3\text{°F}\times 60\ \text{min/hr}$, the
sensible-air factor on a CFM basis.)

**Cooling** — sol-air uplift makes sunlit surfaces conduct as if the outdoor air
were hotter; plus infiltration, window solar gain, internal gains, and a latent
adder:

$$Q_{\text{roof}}=\frac{A_{\text{roof}}\big[(T_{\text{out}}+\Delta T_{\text{sol,roof}})-T_{\text{in}}\big]}{R_{\text{roof}}},\quad
Q_{\text{solar}}=A_{\text{win}}\cdot\text{SHGC}\cdot I_{\text{glass}}$$

$$Q_{\text{sens}}=\sum Q_{\text{cond}}+Q_{\text{inf}}+Q_{\text{solar}}+Q_{\text{int}},\qquad
Q_{\text{cool}}=Q_{\text{sens}}\left(1+\tfrac{\text{latent}\%}{100}\right)$$

with cooling tonnage $Q_{\text{cool}}/12000$.


In [12]:
# ── Thermal loads: heating & cooling design loads ───────────────────────────
import math
BTUH_W = 0.293071

# From the envelope (§11) — recomputed compactly here for a standalone cell
R_WALL, R_ROOF, R_FLOOR, R_WIN, R_DOOR = 12.0, 12.3, 12.5, 3.33, 6.0   # ~design-basis R
A_WALL, A_ROOF, A_FLOOR, A_WIN, A_DOOR = 355.0, 90.0, 90.0, 16.0, 17.8 # ft^2
INT_L, INT_W, INT_H = 130.0, 76.0, 88.0   # interior box (in) after wall build-up

# Design conditions (Austin: 99% heating 28F / cooling 100F), indoor 72F
T_OUT_H, T_OUT_C, T_IN = 28, 100, 72
ACH = 0.5
OCC = 2.0                              # adult-equivalents
OCC_W_SENS, OCC_W_TOTAL = 70, 117      # ASHRAE seated adult (W)
PLUG_W = 300                           # steady internal plug heat (W), from loads table
WINDOW_SHGC, SOLAR_GLASS_BTU = 0.40, 30
LATENT_PCT = 30
UP_ROOF, UP_WALL = 3.0, 3.5            # effective sol-air uplift (F): PV-shaded roof / light walls

vol_ft3 = (INT_L*INT_W*INT_H)/1728

# Heating
dTh = max(0, T_IN - T_OUT_H)
cond_h = dTh*(A_WALL/R_WALL + A_ROOF/R_ROOF + A_FLOOR/R_FLOOR + A_WIN/R_WIN + A_DOOR/R_DOOR)
inf_h  = 1.08 * (ACH*vol_ft3/60) * dTh
heat_cert = cond_h + inf_h
gains_btu = (OCC*OCC_W_SENS + PLUG_W)/BTUH_W
heat_oper = max(0, heat_cert - gains_btu)

# Cooling (sol-air on roof/wall)
c_roof  = A_ROOF*((T_OUT_C+UP_ROOF)-T_IN)/R_ROOF
c_wall  = A_WALL*((T_OUT_C+UP_WALL)-T_IN)/R_WALL
c_floor = A_FLOOR*(T_OUT_C-T_IN)/R_FLOOR
c_win   = A_WIN*(T_OUT_C-T_IN)/R_WIN
c_door  = A_DOOR*(T_OUT_C-T_IN)/R_DOOR
inf_c   = 1.08*(ACH*vol_ft3/60)*max(0, T_OUT_C-T_IN)
solar   = A_WIN*WINDOW_SHGC*SOLAR_GLASS_BTU
intern  = (OCC*OCC_W_TOTAL + PLUG_W)/BTUH_W
sens    = max(0, c_roof+c_wall+c_floor+c_win+c_door+inf_c+solar+intern)
cool    = sens*(1 + LATENT_PCT/100)

print("Thermal design loads (Austin design temps, indoor 72F)")
print(f"  HEATING @ {T_OUT_H}F  (dT {dTh}F)")
print(f"    Conduction        : {cond_h*BTUH_W:6.0f} W")
print(f"    Infiltration      : {inf_h*BTUH_W:6.0f} W  ({ACH} ACH)")
print(f"    Certified load    : {heat_cert*BTUH_W:6.0f} W  (no gain credit)")
print(f"    Operating load    : {heat_oper*BTUH_W:6.0f} W  (gains {gains_btu*BTUH_W:.0f} W credited)")
print(f"  COOLING @ {T_OUT_C}F")
print(f"    Sol-air conduction: {(c_roof+c_wall+c_floor+c_win+c_door)*BTUH_W:6.0f} W  (roof +{UP_ROOF}F, wall +{UP_WALL}F)")
print(f"    Window solar      : {solar*BTUH_W:6.0f} W  (SHGC {WINDOW_SHGC} x {SOLAR_GLASS_BTU})")
print(f"    Internal gains    : {intern*BTUH_W:6.0f} W")
print(f"    Sensible          : {sens*BTUH_W:6.0f} W")
print(f"    Total (+{LATENT_PCT}% latent): {cool*BTUH_W:6.0f} W  =  {cool/12000:.2f} tons")

Thermal design loads (Austin design temps, indoor 72F)
  HEATING @ 28F  (dT 44F)
    Conduction        :    669 W
    Infiltration      :     58 W  (0.5 ACH)
    Certified load    :    727 W  (no gain credit)
    Operating load    :    287 W  (gains 440 W credited)
  COOLING @ 100F
    Sol-air conduction:    462 W  (roof +3.0F, wall +3.5F)
    Window solar      :     56 W  (SHGC 0.4 x 30)
    Internal gains    :    534 W
    Sensible          :   1090 W
    Total (+30% latent):   1417 W  =  0.40 tons


## 13 · HVAC COP &amp; seasonal degree-day energy

**Coefficient of performance** — interpolated from a temperature-vs-COP curve,
derated for portable units, floored at 1 (resistance backup):

$$\text{COP}(T_{\text{out}})=\max\!\big(1,\ \text{interp}(T_{\text{out}})\cdot k\big),\qquad
P_{\text{elec}}=\frac{Q_{\text{thermal}}}{\text{COP}}$$

with $k=0.85$ for a Wave-3-class portable, $k=1$ for a mini-split.

**Seasonal energy** — a variable-base degree-day estimate. The heating threshold
is *this envelope's own balance point* (where free gains cover the losses), not the
generic base-65:

$$UA_{\text{eff}}=UA+1.08\frac{\text{ACH}\cdot V_{\text{ft}^3}}{60},\qquad
T_b=T_{\text{set}}-\frac{Q_{\text{gains}}}{UA_{\text{eff}}}$$

HDD is rescaled from the base-65 normal by a quadratic shape anchored to the winter
mean, then converted to electricity:

$$\text{HDD}(T_b)\approx\text{HDD}_{65}\left(\frac{T_b-T_{\text{win}}}{65-T_{\text{win}}}\right)^2,\qquad
E_{\text{heat}}=\frac{UA_{\text{eff}}\cdot\text{HDD}(T_b)\cdot24}{3412\cdot\text{COP}_h}$$

$$E_{\text{cool}}=\frac{UA_{\text{eff}}\cdot\text{CDD}_{65}\cdot24}{3412}\cdot\frac{Q_{\text{cool,design}}}{Q_{\text{cond,design}}}\cdot\frac{1}{\text{COP}_c}$$


In [13]:
# ── HVAC COP & seasonal degree-day energy ───────────────────────────────────
import math

COP_HEAT = [[-13,1.5],[-5,1.7],[5,1.9],[17,2.3],[30,2.8],[47,3.4],[60,3.8]]
COP_COOL = [[75,2.6],[85,2.2],[95,1.9],[105,1.6],[115,1.4]]
WAVE3_FACTOR = 0.85

def interp(pts, x):
    if x <= pts[0][0]:
        a,b = pts[0],pts[1]; return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])
    for a,b in zip(pts, pts[1:]):
        if x <= b[0]: return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])
    a,b = pts[-2],pts[-1]; return a[1]+(b[1]-a[1])*(x-a[0])/(b[0]-a[0])

def hvac_cop(mode, tout, unit="wave3"):
    if unit == "resist": return 1.0
    raw = interp(COP_COOL if mode=="cooling" else COP_HEAT, tout)
    k = WAVE3_FACTOR if unit == "wave3" else 1
    return max(1, raw*k)

# COP at a couple of conditions
print("HVAC COP (portable Wave-3 class, k=0.85)")
for t in (-13, 17, 47):
    print(f"  heating @ {t:>3}F : COP {hvac_cop('heating',t):.2f}")
for t in (85, 100):
    print(f"  cooling @ {t:>3}F : COP {hvac_cop('cooling',t):.2f}")

# Seasonal energy (Austin: HDD65 1600, CDD65 3000; winter mean 52F, summer mean 84F)
UA, VOL_FT3, ACH = 42.0, 502.0, 0.5
INDOOR_SET_F = 72
GAINS_BTU = 1200          # design-day free gains (people + plugs), Btu/hr
HDD65, CDD65 = 1600, 3000
WIN_MEAN_F, SUM_MEAN_F = 52, 84
COOL_DESIGN, COND_DESIGN = 6500, 4200   # design-day cooling total vs pure-conduction baseline (Btu/hr)

UA_inf = 1.08 * (ACH*VOL_FT3/60)
UA_eff = UA + UA_inf
copH = hvac_cop("heating", WIN_MEAN_F)
copC = hvac_cop("cooling", SUM_MEAN_F)
Tb = INDOOR_SET_F - GAINS_BTU/UA_eff
shape = max(0, min(1.2, (max(0, Tb-WIN_MEAN_F)/max(1, 65-WIN_MEAN_F))**2))
hdd_tb = HDD65 * shape
heat_kwh = UA_eff * hdd_tb * 24/3412 / copH
inflate = max(1, COOL_DESIGN/COND_DESIGN)
cool_kwh = UA_eff * CDD65 * 24/3412 * inflate / copC

print()
print("Seasonal energy (Austin normals)")
print(f"  UA_eff              : {UA_eff:.1f} Btu/hr-F (envelope {UA} + infiltration {UA_inf:.1f})")
print(f"  Balance point       : {Tb:.1f} F  (HDD65 rescaled x{shape:.2f} -> {hdd_tb:.0f} DD)")
print(f"  Heating / year      : {heat_kwh:.0f} kWh  (COP {copH:.2f} @ winter mean)")
print(f"  Cooling / year      : {cool_kwh:.0f} kWh  (CDD {CDD65} x {inflate:.2f} solar/latent, COP {copC:.2f})")
print(f"  HVAC total / year   : {heat_kwh+cool_kwh:.0f} kWh")

HVAC COP (portable Wave-3 class, k=0.85)
  heating @ -13F : COP 1.27
  heating @  17F : COP 1.95
  heating @  47F : COP 2.89
  cooling @  85F : COP 1.87
  cooling @ 100F : COP 1.49

Seasonal energy (Austin normals)
  UA_eff              : 46.5 Btu/hr-F (envelope 42.0 + infiltration 4.5)
  Balance point       : 46.2 F  (HDD65 rescaled x0.00 -> 0 DD)
  Heating / year      : 0 kWh  (COP 3.02 @ winter mean)
  Cooling / year      : 798 kWh  (CDD 3000 x 1.55 solar/latent, COP 1.90)
  HVAC total / year   : 798 kWh
